In [1]:
import os
os.chdir("../")
%pwd

'/mnt/e/Deep_Learning_Object_Detection/MLOPs/pneumonia-segmentation'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class OnnxModelConfig:
    onnx_dir:       Path
    onnx_int8_dir:  Path
    
@dataclass(frozen=True)
class TensorRTEngineConfig:
    engine_dir:  Path

@dataclass(frozen=True)
class EvalDataConfig:
    infer_data_dir: Path
    
@dataclass(frozen=True)
class EvaluationParamsConfig:
    batch_size: int
    workers:    int
    image_size: int
    is_augmentation: bool
    threshold:  float

@dataclass(frozen=True)
class EvaluationConfig:
    root_dir: Path
    onnx:   OnnxModelConfig
    engine: TensorRTEngineConfig
    data:   EvalDataConfig
    eval:   EvaluationParamsConfig

In [ ]:
from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

class ConfigManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH, 
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def _model_slug(self) -> str:
        p = self.params.prepare_base_model_params
        return f"{p.model_name}_{p.encoder}"

    def _get_onnx_model_config(self, onnx_root: Path) -> OnnxModelConfig:
        slug = self._model_slug()
        d    = onnx_root / slug
        
        return OnnxModelConfig(
            onnx_dir        = d / "model.onnx",
            onnx_int8_dir   = d / "model_int8.onnx"
        )
        
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = self.config.evaluation_config
        eval_params = self.params.evaluation_params
        create_directories([eval_config.root_dir])
        
        slug = self._model_slug()
        onnx_root   = Path(self.config.onnx_config.root_dir)
        engine_root = Path(self.config.tensorRT_config.root_dir)

        evaluation_config = EvaluationConfig(
            root_dir = Path(eval_config.root_dir),
            onnx = OnnxModelConfig(
                onnx_dir        = onnx_root / slug / "model.onnx",
                onnx_int8_dir   = onnx_root / slug / "model_int8.onnx"
            ),
            engine = TensorRTEngineConfig(
                engine_dir = engine_root / slug / "model.engine",
            ),
            data = EvalDataConfig(
                infer_data_dir = Path(
                    self.config.data_transformation_config.out_infer_dir
                ),
            ),
            eval = EvaluationParamsConfig(
                batch_size      = eval_params.batch_size,
                workers         = eval_params.workers,
                image_size      = eval_params.image_size,
                is_augmentation = eval_params.is_augmentation,
                threshold       = eval_params.threshold
            )
        )
        
        return evaluation_config

In [8]:
import sys, json, time, numpy as np, torch
import onnxruntime as ort
import segmentation_models_pytorch as smp

from pneumonia_segmentation import logging
from pneumonia_segmentation.exception import CustomException
from pneumonia_segmentation.utils.data_loaders import get_eval_dataloader
from pneumonia_segmentation.utils.metrics.iou import compute_iou

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config       = config
        self.val_loader   = get_eval_dataloader(self.config)
        self.criterion    = self._get_loss_function()
    
    def _load_cpu_onnx_model(self, path: str) -> ort.InferenceSession:
        return ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    
    def _load_cuda_onnx_model(self, path: str) -> ort.InferenceSession:
        return ort.InferenceSession(str(path), providers=["CUDAExecutionProvider"])
    
    def _get_loss_function(self):
        dice = smp.losses.DiceLoss(mode='binary')
        focal = smp.losses.FocalLoss(mode='binary')
        return lambda preds, masks: dice(preds, masks) + focal(preds, masks)
    
    @torch.no_grad()
    def _eval_onnx(self, session: ort.InferenceSession) -> dict:
        start = time.perf_counter()
        total_loss, iou_scores = 0.0, []
        
        for batch in self.val_loader:
            images, masks = batch['image'], batch['mask']
            ort_inputs    = {session.get_inputs()[0].name: images.numpy()}
            outputs       = torch.tensor(session.run(None, ort_inputs)[0])
            total_loss   += self.criterion(outputs, masks).item()
            iou_scores.append(
                compute_iou(outputs, masks, threshold=self.config.eval.threshold))
        
        elapsed = (time.perf_counter() - start) * 1000 / len(self.val_loader)
        return {
            "loss": round(total_loss / len(self.val_loader), 4),
            "iou":  round(float(np.mean(iou_scores)), 4),
            "avg_ms":  round(elapsed, 2),
        }
    
    @torch.no_grad()
    def _eval_tensorrt(self) -> dict | None:
        try:
            import tensorrt as trt
            import pycuda.driver as cuda
            import pycuda.autoinit

            logger  = trt.Logger(trt.Logger.WARNING)
            runtime = trt.Runtime(logger)

            with open(self.config.engine.engine_dir, "rb") as f:
                engine = runtime.deserialize_cuda_engine(f.read())

            context = engine.create_execution_context()
            start   = time.perf_counter()
            total_loss, iou_scores = 0.0, []

            for batch in self.val_loader:
                images, masks = batch['image'], batch['mask']
                images_np     = images.numpy().astype(np.float32)

                # Set input shape first, then query output shape
                context.set_input_shape("input", images_np.shape)
                output_shape = tuple(context.get_tensor_shape("output"))
                output       = np.empty(output_shape, dtype=np.float32)

                input_mem  = cuda.mem_alloc(images_np.nbytes)
                output_mem = cuda.mem_alloc(output.nbytes)

                cuda.memcpy_htod(input_mem, images_np)
                context.execute_v2([int(input_mem), int(output_mem)])
                cuda.memcpy_dtoh(output, output_mem)

                outputs     = torch.tensor(output)
                total_loss += self.criterion(outputs, masks).item()
                iou_scores.append(
                    compute_iou(outputs, masks, threshold=self.config.eval.threshold))
            
            elapsed = (time.perf_counter() - start) * 1000 / len(self.val_loader)
            return {
                "loss":   round(total_loss / len(self.val_loader), 4),
                "iou":    round(float(np.mean(iou_scores)), 4),
                "avg_ms": round(elapsed, 2),
            }

        except ImportError:
            logging.info("TensorRT not available, skipping...")
            return None
        
    @torch.no_grad()
    def validate(self):
        try:
            results = {}
            
            logging.info("Evaluating CPU ONNX FP32...")
            results["cpu_onnx_fp32"] = self._eval_onnx(
                self._load_cpu_onnx_model(str(self.config.onnx.onnx_dir)))
            
            logging.info("Evaluating CUDA ONNX FP32...")
            results["cuda_onnx_fp32"] = self._eval_onnx(
                self._load_cuda_onnx_model(str(self.config.onnx.onnx_dir)))
            
            logging.info("Evaluating CPU ONNX INT8...")
            results["cpu_onnx_int8"] = self._eval_onnx(
                self._load_cpu_onnx_model(str(self.config.onnx.onnx_int8_dir)))
            
            logging.info("Evaluating CUDA ONNX INT8...")
            results["cuda_onnx_int8"] = self._eval_onnx(
                self._load_cuda_onnx_model(str(self.config.onnx.onnx_int8_dir)))
            
            logging.info("Evaluating TensorRT...")
            trt_result = self._eval_tensorrt()
            if trt_result:
                results["tensorrt"] = trt_result
            
            self._print_table(results)
            self._save_metrics(results)
            return results
        
        except Exception as e:
            raise CustomException(e, sys)

    def _print_table(self, results: dict):
        print(f"\n{'Model':<20} {'IoU':>8} {'Loss':>8} {'ms/batch':>10}")
        print("-" * 48)
        for name, metrics in results.items():
            print(f"{name:<20} {metrics['iou']:>8.4f} {metrics['loss']:>8.4f} {metrics.get('avg_ms', 0):>10.2f}")

    def _save_metrics(self, results: dict):
        path = self.config.root_dir / self.config.onnx.onnx_int8_dir.parent.name / "metrics.json"
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump(results, f, indent=4)
        logging.info(f"Metrics saved -> {path} | {results   }")

In [9]:
try:
    config = ConfigManager()
    evaluation = Evaluation(config.get_evaluation_config())
    evaluation.validate()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-12 18:29:57,411: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-12 18:29:57,416: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-12 18:29:57,454: INFO: common: created directory at: artifacts]
[2026-04-12 18:29:57,457: INFO: common: created directory at: artifacts/evaluation]
[2026-04-12 18:29:57,467: INFO: 908466738: Evaluating CPU ONNX FP32...]
[2026-04-12 18:30:03,355: INFO: 908466738: Evaluating CUDA ONNX FP32...]


2026-04-12 18:30:03.842034750 [W:onnxruntime:, session_state.cc:1327 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-04-12 18:30:03.842076266 [W:onnxruntime:, session_state.cc:1329 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


[2026-04-12 18:30:04,657: INFO: 908466738: Evaluating CPU ONNX INT8...]
[2026-04-12 18:30:10,687: INFO: 908466738: Evaluating CUDA ONNX INT8...]


2026-04-12 18:30:10.849066294 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 214 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-04-12 18:30:10.853589633 [W:onnxruntime:, session_state.cc:1327 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-04-12 18:30:10.853600736 [W:onnxruntime:, session_state.cc:1329 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


[2026-04-12 18:30:14,534: INFO: 908466738: Evaluating TensorRT...]

Model                     IoU     Loss   ms/batch
------------------------------------------------
cpu_onnx_fp32          0.6611   0.4092     644.69
cuda_onnx_fp32         0.6611   0.4092      82.30
cpu_onnx_int8          0.6596   0.4207     733.02
cuda_onnx_int8         0.6602   0.4207     766.24
tensorrt               0.6611   0.4092      47.58
[2026-04-12 18:30:15,836: INFO: 908466738: Metrics saved -> artifacts/evaluation/manet_resnet34/metrics.json | {'cpu_onnx_fp32': {'loss': 0.4092, 'iou': 0.6611, 'avg_ms': 644.69}, 'cuda_onnx_fp32': {'loss': 0.4092, 'iou': 0.6611, 'avg_ms': 82.3}, 'cpu_onnx_int8': {'loss': 0.4207, 'iou': 0.6596, 'avg_ms': 733.02}, 'cuda_onnx_int8': {'loss': 0.4207, 'iou': 0.6602, 'avg_ms': 766.24}, 'tensorrt': {'loss': 0.4092, 'iou': 0.6611, 'avg_ms': 47.58}}]
